In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
#Datasets & DataLoader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale(0,1) => normalize(-1,1)
transform = transforms.Compose([
    transforms.ToTensor(), # scale (0, 1)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # normalize (-1, 1)
])

train = CIFAR10(root="/kaggle/working/data", train=True, download=True, transform=transform)
test = CIFAR10(root="/kaggle/working/data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:05<00:00, 28.9MB/s] 


In [11]:
trainloader = DataLoader(train, batch_size=64, shuffle=True)
testloader = DataLoader(train, batch_size=64)

# Build the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()


        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1), # in_channels = 3 coz dimension is (32,32,3) where 3 is RGB
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1), # out_channel = should increase 2x and prev_out = curr_in
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) #flattening
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [7]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) #FP
        loss = criterion(output, labels) #loss fnx
        loss.backward() #BP
        optimizer.step() #update params

        epoch_training_loss += loss.item()

    print(f"epoch{epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")
        

epoch1/10 & loss=1.3752378845001425
epoch2/10 & loss=0.9466244849326361
epoch3/10 & loss=0.751686801538443
epoch4/10 & loss=0.6167497860882288
epoch5/10 & loss=0.5078702020980513
epoch6/10 & loss=0.4206836303634107
epoch7/10 & loss=0.35608181193509064
epoch8/10 & loss=0.29992532878256667
epoch9/10 & loss=0.24892680106393975
epoch10/10 & loss=0.20703487309729657


In [13]:
# Evaluation

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy={correct_labels/total_labels*100}")

accuracy=88.914
